# 03 · Auburn offense vs. Florida defense

**Run first:** `python -m src.matchup` · **Spec:** [`docs/analysis_spec.md`](../docs/analysis_spec.md) sections 6–9 · **Decision memo:** [`docs/go_no_go.md`](../docs/go_no_go.md)

Only 2026 plays describe these teams. The primary sample excludes garbage time. Residuals are measured against the league baseline from notebook 02. Components are shrunk toward zero (the league) by an amount learned from how well two early games predicted the rest of the season in 2021–2025.

A positive Auburn component means Auburn's offense produced more than the situation normally produces. A positive Florida component means Florida's defense allowed more. The matchup edge averages the two, so positive favors Auburn.

In [1]:
import json, sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import config

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 100)
T = config.TABLES_DIR
selection = json.loads((config.OUTPUTS_DIR / "models" / "model_selection.json").read_text())

## 1. How much can two games be trusted?

For every FBS team-season in 2021–2025, we asked how well its Weeks 1–2 residual in a situation predicts its residual for the rest of that season.

- **k** is the number of plays of "league average" that gets blended into a team's own plays. A team cell with n plays keeps n / (n + k) of its raw signal.
- **The 2026 column** is an independent method-of-moments estimate from this season's FBS teams. "inf" means no detectable team differences.

In [2]:
k = pd.read_csv(T / "shrinkage_k.csv")
display(k.round(3))

,metric,side,play_family,down_group,k,team_seasons,median_early_plays,shrink_at_median_early_plays,error_reduction_vs_no_trust,early_late_correlation,k_2026_method_of_moments,teams_2026
0,value,offense,pass,passing downs,281.84,664,22.5,0.074,0.027,0.187,130.547,138
1,value,offense,pass,standard downs,473.15,664,34.0,0.067,0.016,0.142,122.576,138
2,value,offense,run,passing downs,158.49,664,11.0,0.065,0.022,0.140,82.577,129
3,value,offense,run,standard downs,398.11,664,44.0,0.100,0.045,0.217,119.343,138
4,value,defense,pass,passing downs,944.06,664,24.0,0.025,0.004,0.082,336.189,138
5,value,defense,pass,standard downs,841.40,664,31.0,0.036,0.007,0.063,inf,138
6,value,defense,run,passing downs,421.70,664,12.0,0.028,0.004,-0.013,inf,135
7,value,defense,run,standard downs,334.97,664,40.0,0.107,0.051,0.238,125.202,138
8,explosive,offense,pass,passing downs,375.84,664,22.5,0.056,0.016,0.118,239.775,138
9,explosive,offense,pass,standard downs,446.68,664,34.0,0.071,0.023,0.178,107.219,138


**Two games say very little.**
- Early and late residuals correlate at only −0.01 to 0.24 across the 16 situation × metric × side cells.
- The estimated k runs from about 160 to 1,060 plays, so the main-map cells (13–54 plays per team) keep roughly 2–12% of their raw signal.
- The 2026 cross-team estimates are the same order of magnitude or larger (several are infinite).

This is the most important result of the day. Any method that took two-game splits at face value would be publishing noise.

## 2. The primary samples, by game

Raw (unshrunk) residuals per game, for context only. Findings use the shrunk cell components below.

In [3]:
plays = pd.read_parquet(config.PROCESSED_DIR / "plays.parquet")
preds = pd.read_parquet(config.INTERIM_DIR / "predictions.parquet")
d = plays[(plays.season == config.CURRENT_SEASON) & plays.in_team_profile].merge(preds, on="play_key")
rows = []
for side, col, team in (("Auburn offense", "pos_team", config.OFFENSE_TEAM), ("Florida defense", "def_pos_team", config.DEFENSE_TEAM)):
    t = d[d[col] == team]
    for (gid, home, away), g in t.groupby(["game_id", "home_team", "away_team"]):
        rows.append({"unit": side, "game": f"{away} at {home}", "offense spread": g.offense_spread.iloc[0], "plays": len(g),
                     "PPA observed": g.ppa.mean(), "PPA expected": g.value_final_adj.mean(), "PPA residual": (g.ppa - g.value_final_adj).mean(),
                     "explosive observed": g.explosive.mean(), "explosive expected": g.explosive_final_adj.mean()})
display(pd.DataFrame(rows).round(3))

,unit,game,offense spread,plays,PPA observed,PPA expected,PPA residual,explosive observed,explosive expected
0,Auburn offense,Baylor at Auburn,-6.5,75,0.028,0.207,-0.178,0.080,0.082
1,Auburn offense,Southern Miss at Auburn,-32.5,59,0.302,0.449,-0.147,0.051,0.094
2,Florida defense,Florida Atlantic at Florida,25.5,61,0.101,0.053,0.048,0.016,0.050
3,Florida defense,Campbell at Florida,51.5,34,-0.128,-0.053,-0.075,0.059,0.042


- **Auburn's offense fell short of PPA expectations in both games by similar margins:** −0.18 per play against Baylor and −0.15 against Southern Miss. Against Southern Miss, as a 32.5-point favorite, it was expected to produce a lot, and it produced 0.30 PPA per play against an expected 0.45.
- **Auburn's explosive-play shortfall is in the Southern Miss game only:** 5.1% against an expected 9.4%. Against Baylor it matched expectation (8.0% vs 8.2%).
- **Florida's defense was within about 0.08 PPA per play of expectation in both games, in opposite directions:** it allowed slightly more than expected against FAU and slightly less against Campbell.

These are raw two-game numbers. Section 1 shows why they are shrunk heavily before anything is concluded.

## 3. Team profiles: the main-map cells (standard/passing downs × run/pass)

In [4]:
rollup = pd.read_csv(T / "team_cells_rollup.csv")
for metric in ("value", "explosive"):
    print(metric)
    display(rollup[rollup.metric == metric][["team", "play_family", "down_group", "plays", "tier", "observed", "expected", "raw_residual", "shrink_factor", "shrunk_residual"]].round(3))

value


,team,play_family,down_group,plays,tier,observed,expected,raw_residual,shrink_factor,shrunk_residual
0,Auburn,pass,passing downs,25,eligible,0.145,0.439,-0.294,0.081,-0.024
1,Auburn,pass,standard downs,40,eligible,0.196,0.276,-0.081,0.078,-0.006
2,Auburn,run,passing downs,15,eligible,0.404,0.478,-0.075,0.086,-0.006
3,Auburn,run,standard downs,54,eligible,0.045,0.237,-0.192,0.119,-0.023
8,Florida,pass,passing downs,21,eligible,0.120,0.187,-0.066,0.022,-0.001
9,Florida,pass,standard downs,40,eligible,-0.137,-0.067,-0.070,0.045,-0.003
10,Florida,run,passing downs,13,limited,0.436,0.141,0.294,0.030,0.009
11,Florida,run,standard downs,21,eligible,-0.042,-0.078,0.036,0.059,0.002


explosive


,team,play_family,down_group,plays,tier,observed,expected,raw_residual,shrink_factor,shrunk_residual
4,Auburn,pass,passing downs,25,eligible,0.040,0.118,-0.078,0.062,-0.005
5,Auburn,pass,standard downs,40,eligible,0.125,0.126,-0.001,0.082,-0.000
6,Auburn,run,passing downs,15,eligible,0.067,0.055,0.012,0.045,0.001
7,Auburn,run,standard downs,54,eligible,0.037,0.052,-0.015,0.064,-0.001
12,Florida,pass,passing downs,21,eligible,0.048,0.058,-0.010,0.019,-0.000
13,Florida,pass,standard downs,40,eligible,0.050,0.063,-0.013,0.036,-0.000
14,Florida,run,passing downs,13,limited,0.000,0.020,-0.020,0.052,-0.001
15,Florida,run,standard downs,21,eligible,0.000,0.022,-0.022,0.026,-0.001


## 4. Supporting cells (down × distance × run/pass)

These feed the two supporting heatmaps (Visuals 1 and 2). Cells under 8 plays are hidden (blank shrunk residual) and 8–14 are labeled limited.

In [5]:
fine = pd.read_csv(T / "team_cells_fine.csv")
display(fine[fine.metric == "value"].pivot_table(index=["play_family", "down_bin", "distance_bin"], columns="team",
        values=["plays", "raw_residual", "shrunk_residual"], aggfunc="first").round(3))

plays         raw_residual         shrunk_residual        
team                              Auburn Florida       Auburn Florida          Auburn Florida
play_family down_bin distance_bin                                                            
pass        1st      long 8+        25.0    21.0       -0.249   0.011          -0.012   0.000
            2nd      long 8+        14.0     9.0       -0.357  -0.173          -0.017  -0.002
                     medium 4-7      4.0     8.0       -0.944  -0.338             NaN  -0.003
                     short 1-3       3.0     3.0       -0.052  -0.085             NaN     NaN
            3rd/4th  long 8+         7.0     2.0       -0.387  -1.497             NaN     NaN
                     medium 4-7      9.0    13.0        0.197   0.181           0.004   0.002
                     short 1-3       3.0     5.0        1.835   0.148             NaN     NaN
run         1st      long 8+        34.0    13.0       -0.202  -0.102          -0.016  -0.004
                     medium 4-7      1.0     NaN       -0.284     NaN             NaN     NaN
                     short 1-3       NaN     1.0          NaN   0.981             NaN     NaN
            2nd      long 8+         8.0     8.0       -0.193  -0.022          -0.009  -0.000
                     medium 4-7      9.0     1.0       -0.500  -0.915          -0.011     NaN
                     short 1-3       1.0     2.0        0.214  -0.138             NaN     NaN
            3rd/4th  long 8+         4.0     1.0        0.651  -0.160             NaN     NaN
                     medium 4-7      3.0     5.0       -0.728   0.670             NaN     NaN
                     short 1-3       9.0     3.0        0.120   1.036           0.003     NaN

## 5. Matchup findings

Eight findings: four cells × two metrics. The 90% intervals come from a play-level bootstrap within the four games. They're descriptive and don't decide labels.

In [6]:
f = pd.read_csv(T / "matchup_findings.csv")
display(f[["play_family", "down_group", "metric", "auburn_plays", "florida_plays", "auburn_raw_residual", "florida_raw_residual",
           "auburn_shrunk", "florida_shrunk", "edge", "edge_low_90", "edge_high_90", "checks_failed", "failed_checks", "label"]].round(4))

,play_family,down_group,metric,auburn_plays,florida_plays,auburn_raw_residual,florida_raw_residual,auburn_shrunk,florida_shrunk,edge,edge_low_90,edge_high_90,checks_failed,failed_checks,label
0,pass,standard downs,value,40,40,-0.0805,-0.0696,-0.0063,-0.0032,-0.0047,-0.0183,0.0091,2,"4, 6",no clear signal
1,pass,standard downs,explosive,40,40,-0.0013,-0.0132,-0.0001,-0.0005,-0.0003,-0.0038,0.0036,3,"4, 5, 6",no clear signal
2,pass,passing downs,value,25,21,-0.2943,-0.0664,-0.0240,-0.0014,-0.0127,-0.0282,0.0042,0,NaN,no clear signal
3,pass,passing downs,explosive,25,21,-0.0779,-0.0099,-0.0049,-0.0002,-0.0025,-0.0043,-0.0001,0,NaN,no clear signal
4,run,standard downs,value,54,21,-0.1917,0.0361,-0.0229,0.0021,-0.0104,-0.0281,0.0059,0,NaN,no clear signal
5,run,standard downs,explosive,54,21,-0.0153,-0.0225,-0.0010,-0.0006,-0.0008,-0.0020,0.0006,1,6,no clear signal
6,run,passing downs,value,15,13,-0.0749,0.2944,-0.0065,0.0088,0.0012,-0.0201,0.0232,3,"6, 7, 10",no clear signal
7,run,passing downs,explosive,15,13,0.0120,-0.0204,0.0005,-0.0011,-0.0003,-0.0019,0.0027,2,"2, 6",no clear signal


## 6. Robustness checks

Each cell shows whether the edge kept its direction under a check. Check 6 (leave one game out) passes only if all four game-out versions keep direction. Blank means the check doesn't apply.

In [7]:
rc = pd.read_csv(T / "robustness_checks.csv")
rc["finding"] = rc.play_family + " / " + rc.down_group + " / " + rc.metric
grid = rc.dropna(subset=["kept_direction"]).assign(kept=lambda x: x.kept_direction.astype(bool)).groupby(["finding", "check"]).kept.all().unstack("check")
display(grid.replace({True: "kept", False: "FLIPPED"}))
print("Influential plays:")
display(f.assign(finding=f.play_family + " / " + f.down_group + " / " + f.metric)[["finding", "influential_play"]])

check,1,2,3,4,5,6,7,9,10
finding,,,,,,,,,
pass / passing downs / explosive,kept,kept,NaN,kept,kept,kept,kept,kept,kept
pass / passing downs / value,kept,kept,kept,kept,NaN,kept,kept,kept,kept
pass / standard downs / explosive,kept,kept,NaN,FLIPPED,FLIPPED,FLIPPED,kept,kept,kept
pass / standard downs / value,kept,kept,kept,FLIPPED,NaN,FLIPPED,kept,kept,kept
run / passing downs / explosive,kept,FLIPPED,NaN,kept,kept,FLIPPED,kept,kept,kept
run / passing downs / value,kept,kept,kept,kept,NaN,FLIPPED,FLIPPED,kept,FLIPPED
run / standard downs / explosive,kept,kept,NaN,kept,kept,FLIPPED,kept,kept,kept
run / standard downs / value,kept,kept,kept,kept,NaN,kept,kept,kept,kept


Influential plays:


,finding,influential_play
0,pass / standard downs / value,Auburn offense: (07:20) No Huddle-Shotgun #17 B.Brown pass intercepted by #22 R.Bush II at BAY04...
1,pass / standard downs / explosive,Auburn offense: (05:03) No Huddle-Shotgun #17 B.Brown pass incomplete short right to #11 K.Singl...
2,pass / passing downs / value,Auburn offense: (00:13) No Huddle-Shotgun #17 B.Brown pass intercepted by #7 D.Jordan at BAY11 b...
3,pass / passing downs / explosive,Auburn offense: (01:48) No Huddle-Shotgun #17 B.Brown pass incomplete deep left to #4 J.Koger th...
4,run / standard downs / value,Auburn offense: (04:11) No Huddle-Shotgun #23 J.Cobb rush middle for 19 yards gain to the USM04 ...
5,run / standard downs / explosive,Auburn offense: (02:42) No Huddle-Shotgun #29 O.Mabson II rush middle for 6 yards gain to the US...
6,run / passing downs / value,Auburn offense: (05:00) No Huddle-Shotgun #17 B.Brown rush middle for 12 yards gain to the USM47...
7,run / passing downs / explosive,Auburn offense: (04:37) No Huddle-Shotgun #17 B.Brown rush right for 7 yards gain to the USM23 (...


## 7. Overall run and pass edges (gate 5)

In [8]:
display(pd.read_csv(T / "pooled_family_edges.csv").round(4))

,play_family,metric,sample,auburn_shrunk,florida_shrunk,edge
0,pass,value,primary,-0.0131,-0.0026,-0.0078
1,pass,value,garbage_included,-0.0153,0.0022,-0.0065
2,pass,value,giveaways_removed,-0.0047,-0.0010,-0.0028
3,pass,explosive,primary,-0.0019,-0.0004,-0.0012
4,pass,explosive,garbage_included,-0.0020,-0.0007,-0.0014
5,pass,explosive,giveaways_removed,-0.0015,-0.0003,-0.0009
6,run,value,primary,-0.0193,0.0047,-0.0073
7,run,value,garbage_included,-0.0193,-0.0112,-0.0152
8,run,value,giveaways_removed,-0.0116,0.0077,-0.0019
9,run,explosive,primary,-0.0006,-0.0008,-0.0007


## 8. Go or no-go

In [9]:
decision = pd.read_csv(T / "go_no_go.csv")
display(decision)

,item,description,result,detail
0,gate 1,Four 2026 games present and reconciled,True,"scores match official records; plays, pass attempts, interceptions within 1"
1,gate 2,Both models beat the naive baseline on 2025,True,value: True (marginal False); explosive: True (marginal False)
2,gate 3,Explosive model reasonably calibrated on 2025,True,observed/expected 0.95-1.05 and slope 0.85-1.15
3,gate 4,At least two robust eligible findings,True,"3 found: pass on passing downs (value, edge -); pass on passing downs (explosive, edge -); run o..."
4,gate 5,Overall run and pass conclusions stable under garbage-time and giveaway checks,True,NaN
5,no-claims trigger,A model fails to beat naive at all,False,NaN
6,no-claims trigger,Strongest favorable finding driven by one play,True,"strongest favorable finding: run on passing downs, value"
7,no-claims trigger,No favorable finding meets the display threshold,False,NaN
8,decision,NO MATCHUP CLAIMS BEFORE THE GAME,NaN,NaN


**Decision under the frozen rules: no matchup claims before the game.**

- **Gates 1–5 all pass.** The data reconciles, both models beat naive, the explosive model is calibrated, three findings are robust, and the overall run and pass conclusions are stable.
- **All three robust findings point against Auburn:**
  - pass on passing downs (PPA)
  - pass on passing downs (explosive)
  - run on standard downs (PPA)

  They are also tiny after shrinkage (about −0.01 PPA per play), and two of their 90% intervals include zero.
- **The only favorable finding is too weak.** Runs on passing downs (PPA) have 15 and 13 plays and an edge of +0.001. It flips without the Southern Miss game, without the FAU game, without one 12-yard B. Brown run, and under the boosted baseline. It's "no clear signal," and because the strongest favorable finding is driven by one play, the frozen no-claims trigger fires.

The honest summary: through two games, public play-by-play cannot identify a situation where Auburn's offense has an edge over Florida's defense. The most consistent pattern is that Auburn's offense has fallen short of what its situations and point spreads normally produce, in both games, and even that is small once two-game reliability is accounted for.